# Datheon Internship — Task 2
## California Housing Price Prediction: OLS, Ridge & Lasso Regression

**Objective:** Predict median house values, engineer interaction features, impute missing values, inspect multicollinearity, and compare OLS, Ridge (L2), and Lasso (L1) using RMSE, MAE, and R².

The notebook loads the provided classic California housing CSV first, and falls back to Scikit-Learn's built-in California Housing dataset if the CSV cannot be downloaded. These are different dataset variants; the notebook prints which one was loaded.


## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
print("Libraries imported successfully.")

## 2. Load the dataset

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
try:
    housing = pd.read_csv(DATA_URL)
    print("Loaded dataset from the provided CSV URL.")
except Exception as exc:
    print(f"Could not download CSV ({exc}). Falling back to Scikit-Learn's built-in California Housing dataset.")
    bunch = fetch_california_housing(as_frame=True)
    housing = bunch.frame.rename(columns={"MedHouseVal": "median_house_value"})
    print("Loaded Scikit-Learn's built-in California Housing dataset.")

display(housing.head())
print("Dataset shape:", housing.shape)

## 3. Data quality and exploratory analysis

In [ ]:
print("Column types:")
display(housing.dtypes.to_frame("dtype"))
print("\\nMissing values:")
display(housing.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\\nNumeric summary:")
display(housing.describe().T)

target_col = "median_house_value"
if target_col not in housing.columns:
    raise ValueError(f"Target column '{target_col}' was not found.")
y = housing[target_col].copy()
X = housing.drop(columns=[target_col]).copy()
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
valid_target = y.notna()
X, y = X.loc[valid_target].copy(), y.loc[valid_target].copy()

corr_df = X[numeric_features].copy()
corr_df[target_col] = y
plt.figure(figsize=(11, 8))
sns.heatmap(corr_df.corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Correlation Heatmap — Numeric Features and Target")
plt.tight_layout()
plt.show()

## 4. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training target mean:", round(y_train.mean(), 3))
print("Test target mean:", round(y_test.mean(), 3))

## 5. Feature engineering and preprocessing

Numeric columns are median-imputed, expanded with degree-2 polynomial features (including pairwise interactions), and standardized. Categorical columns are imputed with the most frequent value and one-hot encoded. All preprocessing is inside the model pipeline, so it is fitted only on training folds and avoids leakage. Interaction features may increase multicollinearity, which is one reason to compare Ridge and Lasso.

In [ ]:
numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("interactions", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler())
])
categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
transformers = [("numeric", numeric_preprocessor, numeric_features)]
if categorical_features:
    transformers.append(("categorical", categorical_preprocessor, categorical_features))
preprocessor = ColumnTransformer(
    transformers=transformers, remainder="drop", verbose_feature_names_out=False
)
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 6. Define OLS, Ridge, and Lasso models

In [ ]:
ols_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])
ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge())
])
lasso_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Lasso(max_iter=20000, tol=1e-4))
])

alpha_grid = {"regressor__alpha": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
ridge_search = GridSearchCV(
    ridge_pipeline, param_grid=alpha_grid,
    scoring="neg_root_mean_squared_error", cv=cv, n_jobs=-1
)
lasso_search = GridSearchCV(
    lasso_pipeline, param_grid=alpha_grid,
    scoring="neg_root_mean_squared_error", cv=cv, n_jobs=-1
)
print("Pipelines and hyperparameter grid configured.")

## 7. Train and tune

In [ ]:
ols_model.fit(X_train, y_train)
ridge_search.fit(X_train, y_train)
lasso_search.fit(X_train, y_train)
fitted_models = {
    "OLS": ols_model,
    "Ridge": ridge_search.best_estimator_,
    "Lasso": lasso_search.best_estimator_
}
print("Ridge best alpha:", ridge_search.best_params_["regressor__alpha"])
print("Lasso best alpha:", lasso_search.best_params_["regressor__alpha"])
print("Ridge best mean CV RMSE:", round(-ridge_search.best_score_, 4))
print("Lasso best mean CV RMSE:", round(-lasso_search.best_score_, 4))

## 8. Evaluate held-out test set

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R²": r2_score(y_true, y_pred)
    }

predictions = {}
rows = []
for name, model in fitted_models.items():
    pred = model.predict(X_test)
    predictions[name] = pred
    rows.append({"Model": name, **regression_metrics(y_test, pred)})
results_df = pd.DataFrame(rows).set_index("Model").sort_values("RMSE")
display(results_df.style.format({"RMSE": "{:.4f}", "MAE": "{:.4f}", "R²": "{:.4f}"}))
print("Lower RMSE and MAE are better; higher R² is better.")

## 9. Metric comparison

In [ ]:
metrics_long = results_df.reset_index().melt(id_vars="Model", var_name="Metric", value_name="Score")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["RMSE", "MAE", "R²"]):
    sns.barplot(data=metrics_long[metrics_long["Metric"] == metric], x="Model", y="Score", ax=ax)
    ax.set_title(f"{metric} comparison")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)
plt.suptitle("OLS vs Ridge vs Lasso — Test Performance")
plt.tight_layout()
plt.show()

## 10. Actual vs predicted values

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, pred) in zip(axes, predictions.items()):
    ax.scatter(y_test, pred, alpha=0.25, s=12)
    low = min(float(y_test.min()), float(np.min(pred)))
    high = max(float(y_test.max()), float(np.max(pred)))
    ax.plot([low, high], [low, high], linestyle="--")
    ax.set_title(name)
    ax.set_xlabel("Actual median house value")
    ax.set_ylabel("Predicted median house value")
plt.suptitle("Actual vs Predicted Values (test set)")
plt.tight_layout()
plt.show()

## 11. Residual analysis

In [ ]:
residuals = {}
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for col, (name, pred) in enumerate(predictions.items()):
    resid = y_test.to_numpy() - pred
    residuals[name] = resid
    axes[0, col].scatter(pred, resid, alpha=0.25, s=12)
    axes[0, col].axhline(0, linestyle="--")
    axes[0, col].set_title(f"{name}: Residuals vs Predicted")
    axes[0, col].set_xlabel("Predicted value")
    axes[0, col].set_ylabel("Residual (actual - predicted)")
    sns.histplot(resid, kde=True, ax=axes[1, col], bins=40)
    axes[1, col].axvline(0, linestyle="--")
    axes[1, col].set_title(f"{name}: Residual Distribution")
    axes[1, col].set_xlabel("Residual")
plt.tight_layout()
plt.show()

### Interpreting residuals
- Random scatter around zero is generally desirable.
- Funnel shapes may indicate non-constant error variance.
- Patterns can suggest missing non-linear structure or important predictors.
- Skew can indicate systematic errors in some ranges.
Residual plots are diagnostics; they do not alone prove assumptions hold.

## 12. Coefficients and multicollinearity

In [ ]:
fitted_preprocessor = fitted_models["OLS"].named_steps["preprocessor"]
feature_names = fitted_preprocessor.get_feature_names_out()
coef_table = pd.DataFrame({"Feature": feature_names})
for name, model in fitted_models.items():
    coef_table[name] = model.named_steps["regressor"].coef_
display(coef_table.head(20))

coef_long = coef_table.melt(id_vars="Feature", var_name="Model", value_name="Coefficient")
coef_long["AbsCoefficient"] = coef_long["Coefficient"].abs()
top_features = coef_long.groupby("Feature")["AbsCoefficient"].max().nlargest(15).index
plot_df = coef_long[coef_long["Feature"].isin(top_features)]
plt.figure(figsize=(12, 7))
sns.barplot(data=plot_df, y="Feature", x="Coefficient", hue="Model")
plt.title("Coefficient Comparison: Top 15 Features by Absolute Coefficient")
plt.tight_layout()
plt.show()

Interaction and squared terms may correlate with original features and each other. Ridge tends to shrink correlated coefficients, while Lasso can set some coefficients to zero. The correlation heatmap is a useful first check but does not capture every multivariate dependency.

## 13. GridSearchCV alpha curves

In [ ]:
def plot_grid_search(search, title):
    cv_results = pd.DataFrame(search.cv_results_)
    alphas = cv_results["param_regressor__alpha"].astype(float)
    mean_rmse = -cv_results["mean_test_score"]
    std_rmse = cv_results["std_test_score"]
    plt.figure(figsize=(7, 4))
    plt.errorbar(alphas, mean_rmse, yerr=std_rmse, marker="o", capsize=3)
    plt.xscale("log")
    plt.xlabel("Alpha (log scale)")
    plt.ylabel("Mean CV RMSE")
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_grid_search(ridge_search, "Ridge: CV RMSE by Alpha")
plot_grid_search(lasso_search, "Lasso: CV RMSE by Alpha")

## 14. Final summary

In [ ]:
print("Best model by held-out test RMSE:", results_df.index[0])
display(results_df.round(4))
print("Selected Ridge alpha:", ridge_search.best_params_["regressor__alpha"])
print("Selected Lasso alpha:", lasso_search.best_params_["regressor__alpha"])
print(
    "Interpret the actual results above: compare RMSE and MAE for prediction error, "
    "R² for explained variation, and residual plots for systematic patterns. "
    "Do not assume Ridge or Lasso must outperform OLS."
)

## 15. Limitations and reproducibility
- The classic dataset caps the target at a maximum value, limiting accuracy for high-value homes.
- Random train/test splitting may overestimate performance for predictions in new geographic regions.
- Report the dataset source and the metrics actually produced by this notebook.
- Keep the test set untouched during feature selection and hyperparameter tuning.
- Run all cells from top to bottom.

### Suggested GitHub structure
```text
datheon-task-2-california-housing-regression/
├── california_housing_regression.ipynb
├── README.md
└── requirements.txt
```
